## Time Series

In [1]:
import numpy as np
import pandas as pd
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
PREVIOUS_MAX_ROWS = pd.options.display.max_rows
pd.options.display.max_rows = 20
np.set_printoptions(precision=4, suppress=True)

## Date and Time Data Types and Tools

The Python standard library includes data types for date and time data, as well as calendar-related functionality. The datetime, time, and calendar modules are the main places to start. The datetime.datetime type, or simply datetime, is widely used

In [2]:
from datetime import datetime
now = datetime.now()
print(now)
now.year, now.month,
print(now.day)

2024-04-24 16:10:32.579302
24


datetime stores both the date and time down to the microsecond. Timedelta represents the temporal difference between two datetime objects

In [3]:
datetime(2011, 1, 7) 

datetime.datetime(2011, 1, 7, 0, 0)

In [4]:
delta = datetime(2011, 1, 1) - datetime(2008, 1, 1,5)
delta
print(delta.days)
print(delta.seconds)
delta

1095
68400


datetime.timedelta(days=1095, seconds=68400)

the seconds are 24 - 5 which are subtraction of days, when 19 * 60 * 60 you get 68400, what is left.

You can add (or subtract) a timedelta or multiple thereof to a datetime object to
yield a new shifted object

In [5]:
from datetime import timedelta
start = datetime(2011, 1, 7)
print(start + timedelta(12))
start - 2 * timedelta(12)

2011-01-19 00:00:00


datetime.datetime(2010, 12, 14, 0, 0)

In [10]:
from datetime import timedelta
start = datetime(2011, 1, 7)
print(start + timedelta(10))
start - 2 * timedelta(10)

2011-01-17 00:00:00


datetime.datetime(2010, 12, 18, 0, 0)

### Converting Between String and Datetime

You can format datetime objects and pandas Timestamp objects, which I’ll introduce later, as strings using str or the strftime method, passing a format specification

In [11]:
stamp = datetime(2011, 1, 3)
print(str(stamp))
stamp.strftime('%Y-%m-%d')

2011-01-03 00:00:00


'2011-01-03'

In [13]:
stamp.strftime('%d-%m-%Y')

'03-01-2011'

In [14]:
stamp.strftime('%Y-%m-%d-%Y')

'2011-01-03-2011'

You can use these same format codes to convert strings to dates using date time.strptime

In [15]:
value = '2011-01-03'
print(datetime.strptime(value, '%Y-%m-%d'))
datestrs = ['7/6/2011', '8/6/2011']
[datetime.strptime(x, '%m/%d/%Y') for x in datestrs]

2011-01-03 00:00:00


[datetime.datetime(2011, 7, 6, 0, 0), datetime.datetime(2011, 8, 6, 0, 0)]

datetime.strptime is a good way to parse a date with a known format. However, it can be a bit annoying to have to write a format spec each time, especially for common date formats. In this case, you can use the parser.parse method in the third-party dateutil package (this is installed automatically when you install pandas)

In [16]:
from dateutil.parser import parse
parse('2011-01-03')

datetime.datetime(2011, 1, 3, 0, 0)

dateutil is capable of parsing most human-intelligible date representations

In [17]:
parse('Jan 31, 1997 10:45 PM')

datetime.datetime(1997, 1, 31, 22, 45)

In international locales, day appearing before month is very common, so you can pass dayfirst=True to indicate this

In [18]:
parse('6/12/2011', dayfirst=True)

datetime.datetime(2011, 12, 6, 0, 0)

pandas is generally oriented toward working with arrays of dates, whether used as an axis index or a column in a DataFrame. The to_datetime method parses many different kinds of date representations. Standard date formats like ISO 8601 can be parsed very quickly

In [19]:
datestrs = ['2011-07-06 12:00:00', '2011-08-06 00:00:00']
pd.to_datetime(datestrs)

DatetimeIndex(['2011-07-06 12:00:00', '2011-08-06 00:00:00'], dtype='datetime64[ns]', freq=None)

It also handles values that should be considered missing (None, empty string, etc.)

In [20]:
idx = pd.to_datetime(datestrs + [None])
idx

DatetimeIndex(['2011-07-06 12:00:00', '2011-08-06 00:00:00', 'NaT'], dtype='datetime64[ns]', freq=None)

In [21]:
print(idx[2])
pd.isnull(idx)

NaT


array([False, False,  True])

## Time Series Basics

A basic kind of time series object in pandas is a Series indexed by timestamps, which is often represented external to pandas as Python strings or datetime objects

In [23]:
from datetime import datetime
dates = [datetime(2011, 1, 2), datetime(2011, 1, 5),
         datetime(2011, 1, 7), datetime(2011, 1, 8),
         datetime(2011, 1, 10), datetime(2011, 1, 12)]
ts = pd.Series(np.random.randn(6), index=dates)
ts

2011-01-02    0.092908
2011-01-05    0.281746
2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
2011-01-12   -1.296221
dtype: float64

Under the hood, these datetime objects have been put in a DatetimeIndex

In [24]:
ts.index

DatetimeIndex(['2011-01-02', '2011-01-05', '2011-01-07', '2011-01-08',
               '2011-01-10', '2011-01-12'],
              dtype='datetime64[ns]', freq=None)

Like other Series, arithmetic operations between differently indexed time series automatically align on the dates

In [25]:
ts + ts[::2]

2011-01-02    0.185816
2011-01-05         NaN
2011-01-07    1.538045
2011-01-08         NaN
2011-01-10    2.014379
2011-01-12         NaN
dtype: float64

Recall that ts[::2] selects every second element in ts.
pandas stores timestamps using NumPy’s datetime64 data type at the nanosecond
resolution

In [26]:
ts.index.dtype

dtype('<M8[ns]')

Scalar values from a DatetimeIndex are pandas Timestamp objects

In [27]:
stamp = ts.index[0]
stamp

Timestamp('2011-01-02 00:00:00')

A Timestamp can be substituted anywhere you would use a datetime object. Addi‐ tionally, it can store frequency information (if any) and understands how to do time zone conversions and other kinds of manipulations. More on both of these things later

### Indexing, Selection, Subsetting

Time series behaves like any other pandas.Series when you are indexing and selecting data based on label

In [28]:
stamp = ts.index[2]
ts[stamp]

0.7690225676118387

In [29]:
ts['2011-01-07']

0.7690225676118387

In [30]:
ts

2011-01-02    0.092908
2011-01-05    0.281746
2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
2011-01-12   -1.296221
dtype: float64

As a convenience, you can also pass a string that is interpretable as a date

In [31]:
print(ts)
print(ts['1/10/2011'])
ts['20110110']

2011-01-02    0.092908
2011-01-05    0.281746
2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
2011-01-12   -1.296221
dtype: float64
1.0071893575830049


1.0071893575830049

For longer time series, a year or only a year and month can be passed to easily select slices of data

In [32]:
longer_ts = pd.Series(np.random.randn(1000),
                      index=pd.date_range('1/1/2000', periods=1000))
longer_ts

2000-01-01    0.274992
2000-01-02    0.228913
2000-01-03    1.352917
2000-01-04    0.886429
2000-01-05   -2.001637
                ...   
2002-09-22   -0.178098
2002-09-23    2.122315
2002-09-24    0.061192
2002-09-25    0.884111
2002-09-26   -0.608506
Freq: D, Length: 1000, dtype: float64

In [33]:
longer_ts['2002']

2002-01-01    0.513393
2002-01-02    1.641653
2002-01-03    0.580790
2002-01-04   -1.707340
2002-01-05   -0.178355
                ...   
2002-09-22   -0.178098
2002-09-23    2.122315
2002-09-24    0.061192
2002-09-25    0.884111
2002-09-26   -0.608506
Freq: D, Length: 269, dtype: float64

Here, the string '2001' is interpreted as a year and selects that time period. This also
works if you specify the month

In [34]:
longer_ts['2001-05']

2001-05-01    1.489410
2001-05-02    1.264250
2001-05-03   -0.761837
2001-05-04   -0.331617
2001-05-05   -1.751315
                ...   
2001-05-27    1.297622
2001-05-28   -1.686933
2001-05-29    1.089539
2001-05-30    2.060882
2001-05-31   -0.241235
Freq: D, Length: 31, dtype: float64

Slicing with datetime objects works as well

In [35]:
ts

2011-01-02    0.092908
2011-01-05    0.281746
2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
2011-01-12   -1.296221
dtype: float64

In [36]:
ts[datetime(2011, 1, 7):]

2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
2011-01-12   -1.296221
dtype: float64

Because most time series data is ordered chronologically, you can slice with time‐stamps not contained in a time series to perform a range query

In [37]:
ts
ts['1/6/2011':'1/11/2011']

2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
dtype: float64

As before, you can pass either a string date, datetime, or timestamp. Remember that slicing in this manner produces views on the source time series like slicing NumPy arrays. This means that no data is copied and modifications on the slice will be reflected in the original data.
There is an equivalent instance method, truncate, that slices a Series between two dates

In [38]:
ts.truncate(after='1/9/2011')

2011-01-02    0.092908
2011-01-05    0.281746
2011-01-07    0.769023
2011-01-08    1.246435
dtype: float64

All of this holds true for DataFrame as well, indexing on its rows

In [39]:
dates = pd.date_range('1/1/2000', periods=100, freq='W-WED')
long_df = pd.DataFrame(np.random.randn(100, 4),
                       index=dates,
                       columns=['Colorado', 'Texas',
                                'New York', 'Ohio'])
print(long_df)
long_df.loc['5-2001']
long_df.loc['2001-5']

            Colorado     Texas  New York      Ohio
2000-01-05 -0.072052  0.544066  0.323886 -1.683325
2000-01-12  0.526860  1.858791 -0.548419 -0.279397
2000-01-19 -0.021299 -0.287990  0.089175  0.522858
2000-01-26  0.572796 -1.760372  1.128179  1.568606
2000-02-02 -0.342277 -0.009813  0.053072 -0.041943
...              ...       ...       ...       ...
2001-10-31 -1.857016  0.449495 -0.061732  1.233914
2001-11-07  0.705830 -1.309077 -1.537380  0.531551
2001-11-14  2.047573  0.446691 -0.223556  0.092835
2001-11-21  0.716076  0.657198 -0.066748  0.838639
2001-11-28 -0.117388 -0.517795 -0.116696  2.389645

[100 rows x 4 columns]


,Colorado,Texas,New York,Ohio
2001-05-02,0.927335,1.513906,0.538600,1.273768
2001-05-09,0.667876,-0.969206,1.676091,-0.817649
2001-05-16,0.050188,1.951312,3.260383,0.963301
2001-05-23,1.201206,-1.852001,2.406778,0.841176
2001-05-30,-0.749181,-2.989741,-1.295289,-1.690195


### Time Series with Duplicate Indices

In some applications, there may be multiple data observations falling on a particular timestamp. Here is an example

In [40]:
dates = pd.DatetimeIndex(['1/1/2000', '1/2/2000', '1/2/2000',
                          '1/2/2000', '1/3/2000'])
dup_ts = pd.Series(np.arange(5), index=dates)
dup_ts

2000-01-01    0
2000-01-02    1
2000-01-02    2
2000-01-02    3
2000-01-03    4
dtype: int32

We can tell that the index is not unique by checking its is_unique property

In [41]:
dup_ts.index.is_unique

False

Indexing into this time series will now either produce scalar values or slices depending on whether a timestamp is duplicated

In [42]:
print(dup_ts['1/3/2000']) # not duplicated give me value of the series
dup_ts['1/2/2000']  # duplicated

4


2000-01-02    1
2000-01-02    2
2000-01-02    3
dtype: int32

Suppose you wanted to aggregate the data having non-unique timestamps. One way to do this is to use groupby and pass level

In [43]:
grouped = dup_ts.groupby(level=0)
print(dup_ts)
print(grouped.mean())
grouped.count()

2000-01-01    0
2000-01-02    1
2000-01-02    2
2000-01-02    3
2000-01-03    4
dtype: int32
2000-01-01    0.0
2000-01-02    2.0
2000-01-03    4.0
dtype: float64


2000-01-01    1
2000-01-02    3
2000-01-03    1
dtype: int64

## Date Ranges, Frequencies, and Shifting

Generic time series in pandas are assumed to be irregular; that is, they have no fixed frequency. For many applications this is sufficient. However, it’s often desirable to work relative to a fixed frequency, such as daily, monthly, or every 15 minutes, even if that means introducing missing values into a time series. Fortunately pandas has a full suite of standard time series frequencies and tools for resampling, inferring frequencies, and generating fixed-frequency date ranges. For example, you can convert the sample time series to be fixed daily frequency by calling resample

In [44]:
print(ts)
resampler = ts.resample('D')
#print(resampler)
print(resampler.sum())

2011-01-02    0.092908
2011-01-05    0.281746
2011-01-07    0.769023
2011-01-08    1.246435
2011-01-10    1.007189
2011-01-12   -1.296221
dtype: float64
2011-01-02    0.092908
2011-01-03    0.000000
2011-01-04    0.000000
2011-01-05    0.281746
2011-01-06    0.000000
2011-01-07    0.769023
2011-01-08    1.246435
2011-01-09    0.000000
2011-01-10    1.007189
2011-01-11    0.000000
2011-01-12   -1.296221
Freq: D, dtype: float64


In [45]:
index = pd.date_range('1/1/2000', periods=9, freq='T')
series = pd.Series(range(9), index=index)
print(series)
print(series.resample('3T').sum())

2000-01-01 00:00:00    0
2000-01-01 00:01:00    1
2000-01-01 00:02:00    2
2000-01-01 00:03:00    3
2000-01-01 00:04:00    4
2000-01-01 00:05:00    5
2000-01-01 00:06:00    6
2000-01-01 00:07:00    7
2000-01-01 00:08:00    8
Freq: T, dtype: int64
2000-01-01 00:00:00     3
2000-01-01 00:03:00    12
2000-01-01 00:06:00    21
Freq: 3T, dtype: int64


The string 'D' is interpreted as daily frequency.
Conversion between frequencies or resampling is a big enough topic to have its own section later. Here I’ll show you how to use the base frequencies and multiples thereof.

### Generating Date Ranges

While I used it previously without explanation, pandas.date_range is responsible for generating a DatetimeIndex with an indicated length according to a particular frequency

In [46]:
index = pd.date_range('2012-04-01', '2012-06-01')
index

DatetimeIndex(['2012-04-01', '2012-04-02', '2012-04-03', '2012-04-04',
               '2012-04-05', '2012-04-06', '2012-04-07', '2012-04-08',
               '2012-04-09', '2012-04-10', '2012-04-11', '2012-04-12',
               '2012-04-13', '2012-04-14', '2012-04-15', '2012-04-16',
               '2012-04-17', '2012-04-18', '2012-04-19', '2012-04-20',
               '2012-04-21', '2012-04-22', '2012-04-23', '2012-04-24',
               '2012-04-25', '2012-04-26', '2012-04-27', '2012-04-28',
               '2012-04-29', '2012-04-30', '2012-05-01', '2012-05-02',
               '2012-05-03', '2012-05-04', '2012-05-05', '2012-05-06',
               '2012-05-07', '2012-05-08', '2012-05-09', '2012-05-10',
               '2012-05-11', '2012-05-12', '2012-05-13', '2012-05-14',
               '2012-05-15', '2012-05-16', '2012-05-17', '2012-05-18',
               '2012-05-19', '2012-05-20', '2012-05-21', '2012-05-22',
               '2012-05-23', '2012-05-24', '2012-05-25', '2012-05-26',
      

By default, date_range generates daily timestamps. If you pass only a start or end
date, you must pass a number of periods to generate

In [47]:
print(pd.date_range(start='2012-04-01', periods=20))
pd.date_range(end='2012-06-01', periods=20)

DatetimeIndex(['2012-04-01', '2012-04-02', '2012-04-03', '2012-04-04',
               '2012-04-05', '2012-04-06', '2012-04-07', '2012-04-08',
               '2012-04-09', '2012-04-10', '2012-04-11', '2012-04-12',
               '2012-04-13', '2012-04-14', '2012-04-15', '2012-04-16',
               '2012-04-17', '2012-04-18', '2012-04-19', '2012-04-20'],
              dtype='datetime64[ns]', freq='D')


DatetimeIndex(['2012-05-13', '2012-05-14', '2012-05-15', '2012-05-16',
               '2012-05-17', '2012-05-18', '2012-05-19', '2012-05-20',
               '2012-05-21', '2012-05-22', '2012-05-23', '2012-05-24',
               '2012-05-25', '2012-05-26', '2012-05-27', '2012-05-28',
               '2012-05-29', '2012-05-30', '2012-05-31', '2012-06-01'],
              dtype='datetime64[ns]', freq='D')

The start and end dates define strict boundaries for the generated date index. For example, if you wanted a date index containing the last business day of each month, you would pass the 'BM' frequency (business end of month) and only dates falling on or inside the date interval will be included:

In [48]:
pd.date_range('2000-01-01', '2000-12-01', freq='BM')

DatetimeIndex(['2000-01-31', '2000-02-29', '2000-03-31', '2000-04-28',
               '2000-05-31', '2000-06-30', '2000-07-31', '2000-08-31',
               '2000-09-29', '2000-10-31', '2000-11-30'],
              dtype='datetime64[ns]', freq='BM')

date_range by default preserves the time (if any) of the start or end timestamp

In [49]:
pd.date_range('2012-05-02 12:56:31', periods=5)

DatetimeIndex(['2012-05-02 12:56:31', '2012-05-03 12:56:31',
               '2012-05-04 12:56:31', '2012-05-05 12:56:31',
               '2012-05-06 12:56:31'],
              dtype='datetime64[ns]', freq='D')

Sometimes you will have start or end dates with time information but want to generate a set of timestamps normalized to midnight as a convention. To do this, there is a normalize option

In [50]:
pd.date_range('2012-05-02 12:56:31', periods=5, normalize=True)

DatetimeIndex(['2012-05-02', '2012-05-03', '2012-05-04', '2012-05-05',
               '2012-05-06'],
              dtype='datetime64[ns]', freq='D')

### Frequencies and Date Offsets

Frequencies in pandas are composed of a base frequency and a multiplier. Base frequencies are typically referred to by a string alias, like 'M' for monthly or 'H' for hourly. For each base frequency, there is an object defined generally referred to as a date offset. For example, hourly frequency can be represented with the Hour class

In [51]:
from pandas.tseries.offsets import Hour, Minute
hour = Hour()
hour

<Hour>

You can define a multiple of an offset by passing an integer

In [52]:
four_hours = Hour(4)
four_hours

<4 * Hours>

In most applications, you would never need to explicitly create one of these objects, instead using a string alias like 'H' or '4H'. Putting an integer before the base frequency creates a multiple

In [53]:
pd.date_range('2000-01-01', '2000-01-03 23:59', freq='4h')

DatetimeIndex(['2000-01-01 00:00:00', '2000-01-01 04:00:00',
               '2000-01-01 08:00:00', '2000-01-01 12:00:00',
               '2000-01-01 16:00:00', '2000-01-01 20:00:00',
               '2000-01-02 00:00:00', '2000-01-02 04:00:00',
               '2000-01-02 08:00:00', '2000-01-02 12:00:00',
               '2000-01-02 16:00:00', '2000-01-02 20:00:00',
               '2000-01-03 00:00:00', '2000-01-03 04:00:00',
               '2000-01-03 08:00:00', '2000-01-03 12:00:00',
               '2000-01-03 16:00:00', '2000-01-03 20:00:00'],
              dtype='datetime64[ns]', freq='4H')

Many offsets can be combined together by addition

In [54]:
Hour(2) + Minute(30)

<150 * Minutes>

Similarly, you can pass frequency strings, like '1h30min', that will effectively be parsed to the same expression

In [55]:
pd.date_range('2000-01-01', periods=10, freq='1h30min')

DatetimeIndex(['2000-01-01 00:00:00', '2000-01-01 01:30:00',
               '2000-01-01 03:00:00', '2000-01-01 04:30:00',
               '2000-01-01 06:00:00', '2000-01-01 07:30:00',
               '2000-01-01 09:00:00', '2000-01-01 10:30:00',
               '2000-01-01 12:00:00', '2000-01-01 13:30:00'],
              dtype='datetime64[ns]', freq='90T')

Some frequencies describe points in time that are not evenly spaced. For example, 'M' (calendar month end) and 'BM' (last business/weekday of month) depend on the number of days in a month and, in the latter case, whether the month ends on a weekend or not. We refer to these as anchored offsets.

#### Week of month dates

One useful frequency class is “week of month,” starting with WOM. This enables you to get dates like the third Friday of each month

In [56]:
rng = pd.date_range('2012-01-01', '2012-09-01', freq='WOM-3FRI')
list(rng)

[Timestamp('2012-01-20 00:00:00'),
 Timestamp('2012-02-17 00:00:00'),
 Timestamp('2012-03-16 00:00:00'),
 Timestamp('2012-04-20 00:00:00'),
 Timestamp('2012-05-18 00:00:00'),
 Timestamp('2012-06-15 00:00:00'),
 Timestamp('2012-07-20 00:00:00'),
 Timestamp('2012-08-17 00:00:00')]

### Shifting (Leading and Lagging) Data

“Shifting” refers to moving data backward and forward through time. Both Series and DataFrame have a shift method for doing naive shifts forward or backward, leaving the index unmodified

In [57]:
ts = pd.Series(np.random.randn(4),
               index=pd.date_range('1/1/2000', periods=4, freq='M'))
print(ts)
print(ts.shift(2))
ts.shift(-2)

2000-01-31   -0.932454
2000-02-29   -0.229331
2000-03-31   -1.140330
2000-04-30    0.439920
Freq: M, dtype: float64
2000-01-31         NaN
2000-02-29         NaN
2000-03-31   -0.932454
2000-04-30   -0.229331
Freq: M, dtype: float64


2000-01-31   -1.14033
2000-02-29    0.43992
2000-03-31        NaN
2000-04-30        NaN
Freq: M, dtype: float64

When we shift like this, missing data is introduced either at the start or the end of the time series.
A common use of shift is computing percent changes in a time series or multiple time series as DataFrame columns. This is expressed as

ts / ts.shift(1) - 1

Because naive shifts leave the index unmodified, some data is discarded. Thus if the frequency is known, it can be passed to shift to advance the timestamps instead of simply the data

In [58]:
ts.shift(2, freq='M')

2000-03-31   -0.932454
2000-04-30   -0.229331
2000-05-31   -1.140330
2000-06-30    0.439920
Freq: M, dtype: float64

Other frequencies can be passed, too, giving you some flexibility in how to lead and lag the data:

In [59]:
print(ts)
print(ts.shift(3, freq='D'))
ts.shift(1, freq='90T')

2000-01-31   -0.932454
2000-02-29   -0.229331
2000-03-31   -1.140330
2000-04-30    0.439920
Freq: M, dtype: float64
2000-02-03   -0.932454
2000-03-03   -0.229331
2000-04-03   -1.140330
2000-05-03    0.439920
dtype: float64


2000-01-31 01:30:00   -0.932454
2000-02-29 01:30:00   -0.229331
2000-03-31 01:30:00   -1.140330
2000-04-30 01:30:00    0.439920
dtype: float64

In [60]:
ts.shift(3, freq='M')

2000-04-30   -0.932454
2000-05-31   -0.229331
2000-06-30   -1.140330
2000-07-31    0.439920
Freq: M, dtype: float64

In [62]:
print(ts)
print(ts.shift(4, freq='D'))

2000-01-31   -0.932454
2000-02-29   -0.229331
2000-03-31   -1.140330
2000-04-30    0.439920
Freq: M, dtype: float64
2000-02-04   -0.932454
2000-03-04   -0.229331
2000-04-04   -1.140330
2000-05-04    0.439920
dtype: float64


The T here stands for minutes

#### Shifting dates with offsets

The pandas date offsets can also be used with datetime or Timestamp objects

In [63]:
from pandas.tseries.offsets import Day, MonthEnd
now = datetime(2011, 11, 17)
print(now)
now + 3 * Day()

2011-11-17 00:00:00


Timestamp('2011-11-20 00:00:00')

If you add an anchored offset like MonthEnd, the first increment will “roll forward” a
date to the next date according to the frequency rule

In [64]:
print(now + MonthEnd())
now + MonthEnd(2)

2011-11-30 00:00:00


Timestamp('2011-12-31 00:00:00')

In [65]:
now + MonthEnd(3)

Timestamp('2012-01-31 00:00:00')

Anchored offsets can explicitly “roll” dates forward or backward by simply using their rollforward and rollback methods, respectively

In [67]:
print(now)
offset = MonthEnd()
print(offset.rollforward(now))
offset.rollback(now)

2011-11-17 00:00:00
2011-11-30 00:00:00


Timestamp('2011-10-31 00:00:00')

A creative use of date offsets is to use these methods with groupby

In [68]:
ts = pd.Series(np.random.randn(20),
               index=pd.date_range('1/15/2000', periods=20, freq='4d'))
print(ts)
ts.groupby(offset.rollforward).mean()

2000-01-15   -0.823758
2000-01-19   -0.520930
2000-01-23    0.350282
2000-01-27    0.204395
2000-01-31    0.133445
2000-02-04    0.327905
2000-02-08    0.072153
2000-02-12    0.131678
2000-02-16   -1.297459
2000-02-20    0.997747
2000-02-24    0.870955
2000-02-28   -0.991253
2000-03-03    0.151699
2000-03-07    1.266151
2000-03-11   -0.202469
2000-03-15    0.050718
2000-03-19    0.639869
2000-03-23    0.597594
2000-03-27   -0.797246
2000-03-31    0.472879
Freq: 4D, dtype: float64


2000-01-31   -0.131313
2000-02-29    0.015961
2000-03-31    0.272399
dtype: float64

Of course, an easier and faster way to do this is using resample

In [69]:
ts.resample('M').mean()

2000-01-31   -0.131313
2000-02-29    0.015961
2000-03-31    0.272399
Freq: M, dtype: float64

## Time Zone Handling

Working with time zones is generally considered one of the most unpleasant parts of time series manipulation. As a result, many time series users choose to work with time series in coordinated universal time or UTC, which is the successor to Greenwich Mean Time and is the current international standard. Time zones are expressed as offsets from UTC; for example, New York is four hours behind UTC during daylight saving time and five hours behind the rest of the year.
In Python, time zone information comes from the third-party pytz library (installable with pip or conda), which exposes the Olson database, a compilation of world time zone information. This is especially important for historical data because the daylight saving time (DST) transition dates (and even UTC offsets) have been changed numerous times depending on the whims of local governments. In the United States, the DST transition times have been changed many times since 1900!
For detailed information about the pytz library, you’ll need to look at that library’s documentation. As far as this book is concerned, pandas wraps pytz’s functionality so you can ignore its API outside of the time zone names. Time zone names can be found interactively and in the docs

In [70]:
import pytz
pytz.common_timezones[-5:]

['US/Eastern', 'US/Hawaii', 'US/Mountain', 'US/Pacific', 'UTC']

To get a time zone object from pytz, use pytz.timezone

In [71]:
tz = pytz.timezone('America/New_York')
tz

<DstTzInfo 'America/New_York' LMT-1 day, 19:04:00 STD>

### Time Zone Localization and Conversion

By default, time series in pandas are time zone naive. For example, consider the following time series

In [72]:
rng = pd.date_range('3/9/2012 9:30', periods=6, freq='D')
ts = pd.Series(np.random.randn(len(rng)), index=rng)
ts

2012-03-09 09:30:00    0.522356
2012-03-10 09:30:00   -0.546348
2012-03-11 09:30:00   -0.733537
2012-03-12 09:30:00    1.302736
2012-03-13 09:30:00    0.022199
2012-03-14 09:30:00    0.364287
Freq: D, dtype: float64

The index’s tz field is None

In [73]:
print(ts.index.tz)

None


Date ranges can be generated with a time zone set

In [74]:
pd.date_range('3/9/2012 9:30', periods=10, freq='D', tz='UTC')

DatetimeIndex(['2012-03-09 09:30:00+00:00', '2012-03-10 09:30:00+00:00',
               '2012-03-11 09:30:00+00:00', '2012-03-12 09:30:00+00:00',
               '2012-03-13 09:30:00+00:00', '2012-03-14 09:30:00+00:00',
               '2012-03-15 09:30:00+00:00', '2012-03-16 09:30:00+00:00',
               '2012-03-17 09:30:00+00:00', '2012-03-18 09:30:00+00:00'],
              dtype='datetime64[ns, UTC]', freq='D')

Conversion from naive to localized is handled by the tz_localize method

In [75]:
ts
ts_utc = ts.tz_localize('UTC')
ts_utc
ts_utc.index

DatetimeIndex(['2012-03-09 09:30:00+00:00', '2012-03-10 09:30:00+00:00',
               '2012-03-11 09:30:00+00:00', '2012-03-12 09:30:00+00:00',
               '2012-03-13 09:30:00+00:00', '2012-03-14 09:30:00+00:00'],
              dtype='datetime64[ns, UTC]', freq='D')

Once a time series has been localized to a particular time zone, it can be converted to another time zone with tz_convert:

In [76]:
ts_utc.tz_convert('America/New_York')

2012-03-09 04:30:00-05:00    0.522356
2012-03-10 04:30:00-05:00   -0.546348
2012-03-11 05:30:00-04:00   -0.733537
2012-03-12 05:30:00-04:00    1.302736
2012-03-13 05:30:00-04:00    0.022199
2012-03-14 05:30:00-04:00    0.364287
Freq: D, dtype: float64

In the case of the preceding time series, which straddles a DST transition in the Amer ica/New_York time zone, we could localize to EST and convert to, say, UTC or Berlin time:

In [103]:
ts_eastern = ts.tz_localize('America/New_York')
ts_eastern.tz_convert('UTC')
ts_eastern.tz_convert('Europe/Berlin')

2012-03-09 15:30:00+01:00   -0.799376
2012-03-10 15:30:00+01:00    0.845542
2012-03-11 14:30:00+01:00   -0.761659
2012-03-12 14:30:00+01:00    0.314976
2012-03-13 14:30:00+01:00    0.798729
2012-03-14 14:30:00+01:00   -1.481627
dtype: float64

tz_localize and tz_convert are also instance methods on DatetimeIndex

In [104]:
ts.index.tz_localize('Asia/Shanghai')

DatetimeIndex(['2012-03-09 09:30:00+08:00', '2012-03-10 09:30:00+08:00',
               '2012-03-11 09:30:00+08:00', '2012-03-12 09:30:00+08:00',
               '2012-03-13 09:30:00+08:00', '2012-03-14 09:30:00+08:00'],
              dtype='datetime64[ns, Asia/Shanghai]', freq=None)